# 05 Cross Validation and Hyperparameter Optimisation

This notebook evaluates model reliability through cross-validation and improves selected algorithms through hyperparameter optimisation. The objective is to move beyond a single train-test split and estimate how consistently each model performs across different partitions of the data.

Cross-validation, GridSearchCV, and Optuna address different parts of the modelling process. Cross-validation measures stability, GridSearchCV exhaustively evaluates a defined parameter grid, and Optuna performs guided probabilistic search over larger or more flexible spaces. Together, these methods reduce the risk of selecting a model that performs well only by chance.


In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(os.path.abspath('..'))

RANDOM_STATE = 42
SAMPLE_SIZE = 50000

%matplotlib inline
sns.set_theme(style='whitegrid')


In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

from src.hyperopt import grid_search, optuna_study
from src.models import get_models
from src.preprocessing import build_preprocess
from src.train import evaluate_model


In [ ]:
def make_pipeline(df, target, model):
    return Pipeline([
        ('preprocess', build_preprocess(df, target)),
        ('model', model),
    ])


In [ ]:
df = pd.read_csv('../datasets/ratings.csv')
df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

TARGET = 'score'
X_df = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=RANDOM_STATE
)
models = get_models('regression')


## Cross Validation

Cross-validation is necessary because a single train-test split may produce optimistic or pessimistic results depending on which observations happen to fall into each subset. In 5-fold cross-validation, the dataset is divided into five parts; each part is used once as validation while the remaining four parts are used for training. The final mean RMSE estimates average generalisation performance across folds.

This procedure helps protect against overfitting because models must perform well across multiple validation sets rather than only one. It also reduces the risk of accidental data leakage when preprocessing is included inside the pipeline, because each fold fits preprocessing steps only on its training portion. This is especially important for imputation, scaling, and one-hot encoding.

The CV RMSE mean indicates expected prediction error on unseen data, while the CV RMSE standard deviation indicates stability. A low standard deviation means performance is consistent across folds. A high standard deviation suggests that the model is sensitive to the particular training sample, possibly because of class/user heterogeneity, sparse categories, or overfitting. Models with similar mean RMSE should therefore be compared using their standard deviation and training cost as well.


In [ ]:
cv_results = []
with mlflow.start_run(run_name='05_cross_validation'):
    for name, model in models.items():
        pipeline_model = make_pipeline(df, TARGET, model)
        scores = cross_val_score(
            pipeline_model,
            X_df,
            y,
            cv=5,
            scoring='neg_mean_squared_error',
            n_jobs=1,
        )
        rmse_scores = np.sqrt(-scores)
        cv_results.append({
            'model': name,
            'cv_rmse_mean': rmse_scores.mean(),
            'cv_rmse_std': rmse_scores.std(),
        })

df_cv = pd.DataFrame(cv_results).sort_values('cv_rmse_mean')
display(df_cv)


## GridSearch: Ridge, Lasso, K-NN Regressor, and K-NN Classifier

GridSearchCV performs exhaustive search over a predefined set of hyperparameter values. It is appropriate when the number of candidate combinations is small and when each value has a clear interpretation. In this notebook, Ridge and Lasso are tuned through `alpha`, while K-NN is tuned through `n_neighbors` and weighting strategy.

For Ridge and Lasso, the best alpha should be compared with the manual alpha analysis from Notebook 02. If both analyses identify similar alpha ranges, the regularisation behaviour is stable. A low alpha indicates that little shrinkage is needed, while a higher alpha indicates that stronger regularisation improves generalisation. The interpretation follows the bias-variance tradeoff: increasing alpha reduces variance but can increase bias if the model becomes too constrained.

For K-NN regression, the selected K indicates the neighbourhood size that minimises validation RMSE. The K-NN classifier uses the same principle but optimises F1 for the binary high-score target. The best classifier K does not have to match the best regression K because regression averages numeric scores, while classification predicts class membership and depends on the class boundary around `score >= 8`. Differences between the two optimal K values reveal that the continuous and binary tasks have different local structures.


In [ ]:
ridge_grid = {'model__alpha': [0.01, 0.1, 1, 10, 100]}
lasso_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}
knn_grid = {'model__n_neighbors': list(range(3, 22, 2)), 'model__weights': ['uniform', 'distance']}

with mlflow.start_run(run_name='05_grid_search_regression'):
    best_ridge = grid_search(make_pipeline(df, TARGET, models['Ridge']), ridge_grid, X_train_df, y_train)
    best_lasso = grid_search(make_pipeline(df, TARGET, models['Lasso']), lasso_grid, X_train_df, y_train)
    best_knn = grid_search(make_pipeline(df, TARGET, models['KNN']), knn_grid, X_train_df, y_train)

grid_summary = pd.DataFrame([
    {'model': 'Ridge', **evaluate_model('Ridge_grid', best_ridge, X_test_df, y_test)},
    {'model': 'Lasso', **evaluate_model('Lasso_grid', best_lasso, X_test_df, y_test)},
    {'model': 'KNN', **evaluate_model('KNN_grid', best_knn, X_test_df, y_test), 'best_k': best_knn.named_steps['model'].n_neighbors},
])
display(grid_summary)


In [ ]:
from sklearn.model_selection import GridSearchCV

df_class = df.copy()
df_class['high_score'] = (df_class['score'] >= 8).astype(int)
CLASS_TARGET = 'high_score'
X_class = df_class.drop(columns=['score', CLASS_TARGET])
y_class = df_class[CLASS_TARGET]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class, test_size=0.2, random_state=RANDOM_STATE
)

class_pipe = make_pipeline(df_class.drop(columns=['score']), CLASS_TARGET, get_models('classification')['KNNClassifier'])
knn_classifier_grid = GridSearchCV(
    class_pipe,
    {'model__n_neighbors': list(range(3, 22, 2)), 'model__weights': ['uniform', 'distance']},
    cv=5,
    scoring='f1',
    n_jobs=1,
)
knn_classifier_grid.fit(Xc_train, yc_train)
print('Best KNN classifier params:', knn_classifier_grid.best_params_)


## Optuna Optimisation

Optuna differs from GridSearchCV because it does not exhaustively test a fixed grid. Instead, it uses a sequential, probabilistic search strategy that proposes new trials based on previous results. This is useful when hyperparameter spaces are larger, continuous, or expensive to evaluate.

The Optuna search tunes Random Forest, SVR, and MLP models. For Random Forest, parameters such as the number of trees and maximum depth control ensemble capacity and overfitting risk. For SVR, parameters such as `C`, `gamma`, and kernel choice control margin flexibility and non-linear similarity. For MLP, hidden-layer structure and regularisation affect the complexity of the neural network and its ability to generalise.

The Optuna results should be compared with the baseline models from Notebook 02. If optimisation improves RMSE, the baseline defaults were not ideal for this dataset. If improvement is small, the default configuration may already be competitive or the available features may limit achievable accuracy. The final choice should account for both the tuned metric and the computational cost of running multiple trials.


In [ ]:
with mlflow.start_run(run_name='05_optuna'):
    best_rf = optuna_study(make_pipeline(df, TARGET, models['RandomForest']), X_train_df, y_train, n_trials=30)
    best_svr = optuna_study(make_pipeline(df, TARGET, SVR(kernel='rbf')), X_train_df, y_train, n_trials=30)
    best_mlp = optuna_study(make_pipeline(df, TARGET, models['MLP_multi']), X_train_df, y_train, n_trials=30)

optuna_summary = pd.DataFrame([
    {'model': 'RandomForest', **evaluate_model('RF_optuna', best_rf, X_test_df, y_test)},
    {'model': 'SVR', **evaluate_model('SVR_optuna', best_svr, X_test_df, y_test)},
    {'model': 'MLP_multi', **evaluate_model('MLP_optuna', best_mlp, X_test_df, y_test)},
])
display(optuna_summary)
